# 🏎️ F1 Head-to-Head Driver Matchup - Classification Model

This notebook:
1. Loads and merges F1 data (2014+ Hybrid Era)
2. Engineers features for ML
3. Trains a RandomForestClassifier to predict finishing position (1-20)
4. Exports model and cleaned data for Dash app

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


## 1. Load Data

In [2]:
DATA_PATH = '../Data/'

results = pd.read_csv(f'{DATA_PATH}results.csv')
races = pd.read_csv(f'{DATA_PATH}races.csv')
drivers = pd.read_csv(f'{DATA_PATH}drivers.csv')
constructors = pd.read_csv(f'{DATA_PATH}constructors.csv')
qualifying = pd.read_csv(f'{DATA_PATH}qualifying.csv')
pit_stops = pd.read_csv(f'{DATA_PATH}pit_stops.csv')

print(f"Results: {results.shape}")
print(f"Races: {races.shape}")
print(f"Drivers: {drivers.shape}")
print(f"Constructors: {constructors.shape}")
print(f"Qualifying: {qualifying.shape}")
print(f"Pit Stops: {pit_stops.shape}")

Results: (26759, 18)
Races: (1125, 18)
Drivers: (861, 9)
Constructors: (212, 5)
Qualifying: (10494, 9)
Pit Stops: (11371, 7)


## 2. Filter for Hybrid Era (2014+)

In [3]:
races_hybrid = races[races['year'] >= 2014].copy()
hybrid_race_ids = races_hybrid['raceId'].unique()
print(f"Hybrid Era races: {len(hybrid_race_ids)} ({races_hybrid['year'].min()}-{races_hybrid['year'].max()})")

Hybrid Era races: 228 (2014-2024)


## 3. Merge Data

In [4]:
# Filter results for hybrid era
results_hybrid = results[results['raceId'].isin(hybrid_race_ids)].copy()

# Merge with races
df = results_hybrid.merge(
    races_hybrid[['raceId', 'year', 'circuitId', 'name']], 
    on='raceId', how='left'
)

# Merge with drivers
df = df.merge(
    drivers[['driverId', 'driverRef', 'forename', 'surname']], 
    on='driverId', how='left'
)
df['driver_name'] = df['forename'] + ' ' + df['surname']

# Merge with constructors
df = df.merge(
    constructors[['constructorId', 'name']], 
    on='constructorId', how='left', suffixes=('', '_constructor')
)
df.rename(columns={'name_constructor': 'constructor_name'}, inplace=True)

print(f"After base merges: {df.shape}")

After base merges: (4626, 26)


In [5]:
# Merge qualifying position
quali_hybrid = qualifying[qualifying['raceId'].isin(hybrid_race_ids)].copy()
quali_subset = quali_hybrid[['raceId', 'driverId', 'position']].copy()
quali_subset.rename(columns={'position': 'qualifying_position'}, inplace=True)

df = df.merge(quali_subset, on=['raceId', 'driverId'], how='left')
print(f"After qualifying merge: {df.shape}")

After qualifying merge: (4626, 27)


In [6]:
# Calculate average pit stop time per race/driver
pit_stops_hybrid = pit_stops[pit_stops['raceId'].isin(hybrid_race_ids)].copy()

pit_agg = pit_stops_hybrid.groupby(['raceId', 'driverId']).agg(
    avg_pit_stop_ms=('milliseconds', 'mean')
).reset_index()

df = df.merge(pit_agg, on=['raceId', 'driverId'], how='left')
print(f"After pit stop merge: {df.shape}")

After pit stop merge: (4626, 28)


## 4. Feature Engineering

In [7]:
# Handle missing values
median_pit = df['avg_pit_stop_ms'].median()
df['avg_pit_stop_ms'] = df['avg_pit_stop_ms'].fillna(median_pit)

df['qualifying_position'] = df['qualifying_position'].fillna(df['grid'])
median_quali = df['qualifying_position'].median()
df['qualifying_position'] = df['qualifying_position'].fillna(median_quali)

print(f"Median pit stop: {median_pit/1000:.2f}s")
print(f"Median quali position: {median_quali}")

Median pit stop: 24.25s
Median quali position: 11.0


In [8]:
# Create feature set
feature_cols = ['driverId', 'constructorId', 'circuitId', 'grid', 'qualifying_position', 'avg_pit_stop_ms']
target_col = 'positionOrder'

# Filter to valid positions (1-20) for classification
df_ml = df[feature_cols + [target_col]].copy()
df_ml = df_ml.dropna()
df_ml = df_ml[(df_ml[target_col] >= 1) & (df_ml[target_col] <= 20)]
df_ml[target_col] = df_ml[target_col].astype(int)

print(f"ML Dataset: {df_ml.shape}")
print(f"Position classes: {sorted(df_ml[target_col].unique())}")

ML Dataset: (4553, 7)
Position classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]


## 5. Train RandomForestClassifier

In [9]:
X = df_ml[feature_cols]
y = df_ml[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Train: 3642 | Test: 911


In [10]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

print("Training classifier...")
clf.fit(X_train, y_train)
print("Done!")

Training classifier...
Done!


In [11]:
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.3f}")
print(f"\nWithin 1 position: {np.mean(np.abs(y_test - y_pred) <= 1):.3f}")
print(f"Within 3 positions: {np.mean(np.abs(y_test - y_pred) <= 3):.3f}")

Test Accuracy: 0.155

Within 1 position: 0.390
Within 3 positions: 0.671


In [12]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)
print(importance)

               feature  importance
5      avg_pit_stop_ms    0.245693
2            circuitId    0.185225
0             driverId    0.164910
4  qualifying_position    0.145376
3                 grid    0.140007
1        constructorId    0.118789


## 6. Save Artifacts

In [13]:
# Save model
joblib.dump(clf, f'{DATA_PATH}rf_classifier_h2h.pkl')
print(f"✅ Model saved to {DATA_PATH}rf_classifier_h2h.pkl")

# Save model info
model_info = {
    'feature_columns': feature_cols,
    'median_pit_stop': median_pit,
    'median_quali': median_quali
}
joblib.dump(model_info, f'{DATA_PATH}model_info.pkl')
print(f"✅ Model info saved")

✅ Model saved to ../Data/rf_classifier_h2h.pkl
✅ Model info saved


In [14]:
# Save cleaned dataset
cols_to_save = [
    'raceId', 'year', 'circuitId', 'name',
    'driverId', 'driver_name', 'driverRef',
    'constructorId', 'constructor_name',
    'grid', 'positionOrder', 'points',
    'qualifying_position', 'avg_pit_stop_ms'
]

df_clean = df[cols_to_save].copy()
df_clean.to_csv(f'{DATA_PATH}f1_cleaned.csv', index=False)
print(f"✅ Cleaned data saved: {df_clean.shape}")

✅ Cleaned data saved: (4626, 14)


In [15]:
# Save lookup tables for dropdowns
drivers_lookup = drivers[drivers['driverId'].isin(df['driverId'].unique())][['driverId', 'forename', 'surname']].copy()
drivers_lookup['full_name'] = drivers_lookup['forename'] + ' ' + drivers_lookup['surname']
drivers_lookup.to_csv(f'{DATA_PATH}drivers_lookup.csv', index=False)

constructors_lookup = constructors[constructors['constructorId'].isin(df['constructorId'].unique())][['constructorId', 'name']].copy()
constructors_lookup.to_csv(f'{DATA_PATH}constructors_lookup.csv', index=False)

circuits = pd.read_csv(f'{DATA_PATH}circuits.csv')
circuits_lookup = circuits[circuits['circuitId'].isin(df['circuitId'].unique())][['circuitId', 'name', 'country']].copy()
circuits_lookup.to_csv(f'{DATA_PATH}circuits_lookup.csv', index=False)

print(f"✅ Lookups saved: {len(drivers_lookup)} drivers, {len(constructors_lookup)} constructors, {len(circuits_lookup)} circuits")

✅ Lookups saved: 59 drivers, 20 constructors, 32 circuits


In [16]:
print("\n" + "="*50)
print("✅ ALL ARTIFACTS SAVED - READY FOR DASH APP")
print("="*50)


✅ ALL ARTIFACTS SAVED - READY FOR DASH APP
